In [16]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [17]:
import json
import pandas as pd
from pathlib import Path


In [18]:
CAREER_LADDER_PATH = "career_path/career_ladders.json"

with open(CAREER_LADDER_PATH, "r") as f:
    CAREER_LADDERS = json.load(f)

CAREER_LADDERS


{'SOFTWARE_ENGINEERING': ['INTERN_SE', 'JR_SE', 'SE', 'SR_SE', 'TECH_LEAD'],
 'QUALITY_ASSURANCE': ['INTERN_QA',
  'JR_QA_EN',
  'QA_EN',
  'SR_QA_EN',
  'QA_LEAD'],
 'DATA_ENGINEERING': ['JR_DATA_EN', 'DATA_EN', 'SR_DATA_EN', 'DATA_ARCH'],
 'AI_ML': ['AI_ML_ENGINEER_INT', 'AI_ML_ENGINEER', 'SENIOR_AI_ML_ENGINEER'],
 'DATA': ['DATA_ANALYST_INT', 'DATA_ENGINEER_INT', 'SENIOR_DATA_ENGINEER']}

In [19]:
ROLE_PROFILE_PATH = "skill_gap/role_skill_profiles.csv"
role_profiles = pd.read_csv(ROLE_PROFILE_PATH)

# sanity check
sorted(role_profiles["role_id"].unique())


['AI_ML_ENGINEER_INT',
 'DATA_ANALYST_INT',
 'DATA_ENGINEER_INT',
 'DEVOPS_TRAINEE',
 'INTERN_SE',
 'JR_BE_DEV',
 'JR_BUSINESS_ANALYST',
 'JR_FE_DEV',
 'JR_FS_DEV',
 'JR_IT_SUPPORT',
 'JR_MOBILE_DEV',
 'JR_QA_ENG',
 'JR_SE',
 'JR_SYS_ADMIN',
 'JR_UI_UX_DESIGNER']

In [20]:
def detect_skill_gap(
    user_skill_ids: set,
    target_role_id: str,
    importance_threshold: float = 0.02
):
    role_df = role_profiles[role_profiles["role_id"] == target_role_id]

    if role_df.empty:
        raise ValueError(f"No role profile found for role_id={target_role_id}")

    required_skills = set(
        role_df[role_df["importance"] >= importance_threshold]["skill_id"]
    )

    missing_skills = sorted(required_skills - user_skill_ids)
    matched_skills = sorted(required_skills & user_skill_ids)

    readiness = (
        len(matched_skills) / len(required_skills)
        if required_skills else 0.0
    )

    return {
        "target_role": target_role_id,
        "readiness_score": round(readiness, 3),
        "missing_skills": missing_skills,
        "matched_skills": matched_skills
    }


In [21]:
def get_next_role(domain: str, current_role: str):
    ladder = CAREER_LADDERS.get(domain)

    if not ladder:
        raise ValueError(f"Unknown domain: {domain}")

    if current_role not in ladder:
        raise ValueError(f"{current_role} not found in domain ladder")

    idx = ladder.index(current_role)

    if idx + 1 >= len(ladder):
        return None  # already at top

    return ladder[idx + 1]


In [22]:
valid_roles = set(role_profiles["role_id"].unique())
valid_roles


{'AI_ML_ENGINEER_INT',
 'DATA_ANALYST_INT',
 'DATA_ENGINEER_INT',
 'DEVOPS_TRAINEE',
 'INTERN_SE',
 'JR_BE_DEV',
 'JR_BUSINESS_ANALYST',
 'JR_FE_DEV',
 'JR_FS_DEV',
 'JR_IT_SUPPORT',
 'JR_MOBILE_DEV',
 'JR_QA_ENG',
 'JR_SE',
 'JR_SYS_ADMIN',
 'JR_UI_UX_DESIGNER'}

In [23]:
def filter_ladders_to_valid_roles(ladders, valid_roles):
    filtered = {}
    for domain, roles in ladders.items():
        filtered_roles = [r for r in roles if r in valid_roles]
        if len(filtered_roles) >= 1:
            filtered[domain] = filtered_roles
    return filtered

CAREER_LADDERS = filter_ladders_to_valid_roles(
    CAREER_LADDERS,
    valid_roles
)

CAREER_LADDERS


{'SOFTWARE_ENGINEERING': ['INTERN_SE', 'JR_SE'],
 'AI_ML': ['AI_ML_ENGINEER_INT'],
 'DATA': ['DATA_ANALYST_INT', 'DATA_ENGINEER_INT']}

In [24]:
def simulate_career_path(
    domain: str,
    current_role: str,
    user_skill_ids: set,
    importance_threshold: float = 0.02
):
    next_role = get_next_role(domain, current_role)

    if next_role is None:
        return {
            "current_role": current_role,
            "message": "You are already at the highest role in this career path."
        }

    gap_result = detect_skill_gap(
        user_skill_ids=user_skill_ids,
        target_role_id=next_role,
        importance_threshold=importance_threshold
    )

    return {
        "domain": domain,
        "current_role": current_role,
        "next_role": next_role,
        "readiness_score": gap_result["readiness_score"],
        "missing_skills": gap_result["missing_skills"],
        "matched_skills": gap_result["matched_skills"]
    }


In [25]:
user_skills = {
    "SK004",  # python
    "SK031",  # ml core
    "SK003"   # sql
}

simulate_career_path(
    domain="AI_ML",
    current_role="AI_ML_ENGINEER_INT",
    user_skill_ids=user_skills
)


{'current_role': 'AI_ML_ENGINEER_INT',
 'message': 'You are already at the highest role in this career path.'}